# Model Context Protocol

Read the [guide](README.md). Run cells in order with the notebooks and apps groups. Offline replay uses real SDK and local protocol paths; it does not measure live model quality.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["MEM0_TELEMETRY"] = "false"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_stdio_client

Predict the outcome, then run the [script](examples/01_stdio_client.py).


In [ ]:
"""Stdio client: Nimbus Tier 3 lesson.

Run: uv run python modules/14_mcp/examples/01_stdio_client.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

from pathlib import Path
from tempfile import TemporaryDirectory

from strands import Agent

from common.lessons import complete, lesson_settings
from common.mcp_clients import reviewed_tools, stdio_booking
from common.models import get_model
from common.pretty import run_cli
from common.testing import tool_message


def main():
    _, cfg = lesson_settings()
    with TemporaryDirectory() as directory:
        with stdio_booking(Path(directory) / "mcp.db", allowed=["search_flights"]) as client:
            tools = reviewed_tools(client, ["nimbus_search_flights"])
            agent = Agent(
                model=get_model(
                    cfg,
                    responses=[
                        tool_message("nimbus_search_flights", {"destination": "Porto"}),
                        "Synthetic Porto flights found.",
                    ],
                ),
                tools=tools,
                callback_handler=None,
                retry_strategy=None,
            )
            result = agent("Find flights to Porto", limits={"turns": 3})
            print("Reviewed tools:", [t.tool_name for t in tools])
            complete(result, cfg)
        # Client and server are closed before the database directory is removed.


run_lesson(main, "modules/14_mcp/examples/01_stdio_client.py")

## 02_http_client

Predict the outcome, then run the [script](examples/02_http_client.py).


In [ ]:
"""Http client: Nimbus Tier 3 lesson.

Run: uv run python modules/14_mcp/examples/02_http_client.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

from pathlib import Path
from tempfile import TemporaryDirectory

from strands import Agent

from common.lessons import complete, lesson_settings
from common.mcp_clients import http_booking, reviewed_tools
from common.models import get_model
from common.pretty import run_cli
from common.testing import tool_message


def main():
    _, cfg = lesson_settings()
    with TemporaryDirectory() as directory:
        with http_booking(Path(directory) / "mcp.db") as (client, url):
            tools = reviewed_tools(client, ["nimbus_search_flights"])
            agent = Agent(
                model=get_model(
                    cfg,
                    responses=[
                        tool_message("nimbus_search_flights", {"destination": "Porto"}),
                        "Synthetic Porto flights found.",
                    ],
                ),
                tools=tools,
                callback_handler=None,
                retry_strategy=None,
            )
            result = agent("Find flights to Porto", limits={"turns": 3})
            print("Reviewed tools:", [t.tool_name for t in tools])
            complete(result, cfg)
        # Client and server are closed before the database directory is removed.


run_lesson(main, "modules/14_mcp/examples/02_http_client.py")

## 03_booking_server

Predict the outcome, then run the [script](examples/03_booking_server.py).


In [ ]:
"""Booking server: Nimbus Tier 3 lesson.

Run: uv run python modules/14_mcp/examples/03_booking_server.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

import argparse
from pathlib import Path
from tempfile import TemporaryDirectory

from common.config import ROOT
from common.lessons import complete, lesson_settings
from common.mcp_clients import stdio_booking
from common.mcp_server import booking_server
from common.pretty import run_cli


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--serve", action="store_true")
    args, _ = lesson_settings(parser)
    if args.serve:
        booking_server(ROOT / ".nimbus/mcp_bookings.db").run(transport="stdio")
    else:
        with TemporaryDirectory() as directory:
            with stdio_booking(Path(directory) / "mcp.db") as client:
                args = {"flight_id": "FLT-0001", "idempotency_key": "demo-booking"}
                first = client.call_tool_sync("one", "create_booking", args)
                second = client.call_tool_sync("two", "create_booking", args)
                assert first["status"] == second["status"] == "success"
                print(first, second)
        complete()


run_lesson(main, "modules/14_mcp/examples/03_booking_server.py")

## 04_aws_servers

Predict the outcome, then run the [script](examples/04_aws_servers.py).


In [ ]:
"""Ask a bounded Lambda cost question; optionally connect pinned AWS MCP servers.

Run: uv run python modules/14_mcp/examples/04_aws_servers.py --mock
Expected: illustrative $1.87 estimate, Lesson complete. Offline cost $0.
Live inference uses configured rates. AWS resources created: none.
"""

import argparse
import json

from common.aws_mcp import aws_server
from common.config import ROOT
from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.pricing_agent import lambda_cost_question


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--connect", choices=["documentation", "pricing"])
    args, cfg = lesson_settings(parser)
    config = json.loads((ROOT / "modules/14_mcp/aws_servers.json").read_text())
    print("Reviewed launch definitions:", config)
    if args.connect and not args.mock:
        with aws_server(cfg, args.connect) as client:
            print("Discovered:", [t.tool_name for t in client.list_tools_sync()])
            if args.connect == "pricing":
                complete(lambda_cost_question(cfg, client=client), cfg)
            else:
                print(
                    client.call_tool_sync(
                        "lambda-docs",
                        "read_documentation",
                        {
                            "url": "https://docs.aws.amazon.com/lambda/latest/dg/welcome.html",
                            "max_length": 5000,
                        },
                    )
                )
                complete()
    else:
        complete(lambda_cost_question(cfg), cfg)


run_lesson(main, "modules/14_mcp/examples/04_aws_servers.py")

## 05_tool_filtering

Predict the outcome, then run the [script](examples/05_tool_filtering.py).


In [ ]:
"""Tool filtering: Nimbus Tier 3 lesson.

Run: uv run python modules/14_mcp/examples/05_tool_filtering.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

from pathlib import Path
from tempfile import TemporaryDirectory

from strands import Agent

from common.lessons import complete, lesson_settings
from common.mcp_clients import reviewed_tools, stdio_booking
from common.models import get_model
from common.pretty import run_cli
from common.testing import tool_message


def main():
    _, cfg = lesson_settings()
    with TemporaryDirectory() as directory:
        with stdio_booking(Path(directory) / "mcp.db", allowed=["search_flights"]) as client:
            tools = reviewed_tools(client, ["nimbus_search_flights"])
            with stdio_booking(
                Path(directory) / "other.db", prefix="other", allowed=["get_booking"]
            ) as other:
                additional = reviewed_tools(other, ["other_get_booking"])
                print("Separate namespace:", [t.tool_name for t in additional])
                agent = Agent(
                    model=get_model(
                        cfg,
                        responses=[
                            tool_message("nimbus_search_flights", {"destination": "Porto"}),
                            "Synthetic Porto flights found.",
                        ],
                    ),
                    tools=[*tools, *additional],
                    callback_handler=None,
                    retry_strategy=None,
                )
                result = agent("Find flights to Porto", limits={"turns": 3})
                print("Reviewed tools:", [t.tool_name for t in tools])
                complete(result, cfg)
        # Client and server are closed before the database directory is removed.


run_lesson(main, "modules/14_mcp/examples/05_tool_filtering.py")

## mcp_booking_assistant

Predict the outcome, then run the [script](use_case/mcp_booking_assistant.py).


In [ ]:
"""booking assistant: Nimbus Tier 3 lesson.

Run: uv run python modules/14_mcp/use_case/mcp_booking_assistant.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

import argparse
from pathlib import Path
from tempfile import TemporaryDirectory

from strands import Agent

from common.lessons import complete, lesson_settings
from common.mcp_clients import reviewed_tools, stdio_booking
from common.models import get_model
from common.pretty import run_cli
from common.pricing_agent import lambda_cost_question
from common.testing import tool_message


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--pricing-live", action="store_true")
    args, cfg = lesson_settings(parser)
    with TemporaryDirectory() as directory:
        with stdio_booking(Path(directory) / "bookings.db") as client:
            tools = reviewed_tools(
                client, ["nimbus_search_flights", "nimbus_create_booking", "nimbus_get_booking"]
            )
            agent = Agent(
                model=get_model(
                    cfg,
                    responses=[
                        tool_message("nimbus_search_flights", {"destination": "Porto"}, "search"),
                        tool_message(
                            "nimbus_create_booking",
                            {"flight_id": "FLT-0001", "idempotency_key": "trip-demo"},
                            "reserve",
                        ),
                        "A local synthetic reservation was created; no airline purchase occurred.",
                    ],
                ),
                tools=tools,
                callback_handler=None,
                retry_strategy=None,
                system_prompt=(
                    "Search then book one synthetic flight. Use idempotency key trip-demo."
                ),
            )
            complete(agent("Find and reserve a Porto flight", limits={"turns": 4}), cfg)
    if args.pricing_live and not args.mock:
        from common.aws_mcp import aws_server

        with aws_server(cfg, "pricing") as client:
            complete(lambda_cost_question(cfg, client=client), cfg)
    else:
        print("Pricing question uses illustrative evidence; no AWS server launched.")
        complete(lambda_cost_question(cfg), cfg)


run_lesson(main, "modules/14_mcp/use_case/mcp_booking_assistant.py")

## Practice

Complete the [exercises](README.md#exercises) and [quiz](quiz.md), then run the module tests.
